## 1) Imports and helpers

In [11]:
# Core imports
import os
import itertools
from datetime import datetime
import pandas as pd
import jax
import jax.numpy as jnp

# Import the framework helpers used by the CLI runner
from framework.registry import ComparisonRegistry
from framework.runner import run_adapter_benchmark
from framework.adapters import setup_bbob_instances

# toml loader (py3.11 has tomllib; otherwise tomli)
try:
    import tomllib
except Exception:
    import tomli as tomllib  # type: ignore

print(f'JAX: {jax.__version__} | Backend: {jax.default_backend()} | Device: {jax.devices()[0].device_kind}')

JAX: 0.8.0 | Backend: cpu | Device: cpu


## 2) Experiment configuration (interactive)
Adjust values here instead of passing a CLI config file.

In [ ]:
# Experiment metadata
EXP_NAME = 'notebook_benchmark'
OUTPUT_DIR = 'results/notebook_runs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Grid-like configuration (similar to CLI's toml grid)
# Print available algorithms for user reference
print('Available algorithms:', list(ComparisonRegistry._registry.keys()))
grid = {
    'algorithms': ['Standard_GA'],  # must match keys in ComparisonRegistry
    'tasks': ['rastrigin'],
    'dimensions': [20],
    'pop_sizes': [100],
    'unroll_factors': [1, 25],
    'generations': 100,
    'repeats': 10,  # lower default for quick notebook runs
    'seeds': [42],
}

# Optional hyperparams that will be merged with algorithm defaults
hyperparams = {}

print('Configured experiment grid:')
for k, v in grid.items():
    print(f'  {k}: {v}')

AttributeError: type object 'ComparisonRegistry' has no attribute 'registry'

## 3) Build job queue
This creates the tuple list the CLI would iterate over. You can subset or preview it before running.

In [ ]:
# Build the job queue (cartesian product)
job_queue = list(itertools.product(
    grid['algorithms'], grid['tasks'], grid['dimensions'], grid['pop_sizes'], grid.get('unroll_factors', [1])
))
print(f'Total configurations: {len(job_queue)}')
# Preview first few
job_queue[:5]

Total configurations: 4


[('malthusjax', 'rastrigin', 20, 100, 1),
 ('malthusjax', 'rastrigin', 20, 100, 25),
 ('evosax', 'rastrigin', 20, 100, 1),
 ('evosax', 'rastrigin', 20, 100, 25)]

## 4) Single-job runner function
Encapsulates the logic from `benchmarks/cli.py` for one (algo,task,dim,pop,unroll) tuple.

In [ ]:
def run_job(algo_key, task, dim, pop, unroll, master_seed, generations, repeats, hypers):
    """Run both adapters for the given job and return packaged result dicts.
    Returns a list with two dicts (one per framework as packaged for CSV).
    """
    spec = ComparisonRegistry.get(algo_key)
    # Merge default hypers with provided ones
    merged_hypers = {**spec.default_hypers, **(hypers or {})}

    # Setup problem instances (MalthusJAX / Evosax adapters expect different objects)
    m_eval, e_prob = setup_bbob_instances(task, dim, master_seed)

    # Factories from registry create adapters given hyperparams etc.
    m_adapter = spec.malthus_factory(pop, dim, master_seed, merged_hypers, m_eval)
    e_adapter = spec.evosax_factory(pop, dim, master_seed, merged_hypers, e_prob)

    # Run benchmarks (returns a small result object from runner)
    res_m = run_adapter_benchmark(m_adapter, generations, master_seed, 'MalthusJAX', pop, unroll, repeats)
    res_e = run_adapter_benchmark(e_adapter, generations, master_seed, 'Evosax', pop, unroll, repeats)

    base = {
        'Algorithm': algo_key, 'Task': task, 'Dim': dim, 'Pop_Size': pop,
        'Unroll': unroll, 'Gens': generations
    }

    def package(res):
        return {
            **base,
            'Framework': res.framework,
            'Mean_GPS': getattr(res, 'mean_gps', None),
            'Mean_Time': getattr(res, 'mean_exec_time', None),
            'Compile_Time': getattr(res, 'compile_time', None),
            'Best_Fitness': getattr(res, 'best_fitness_final', None),
        }

    return [package(res_m), package(res_e)]

## 5) Execute the queue (interactive run)
Control the number of jobs to run so the notebook stays responsive. Start with a subset for learning.

In [ ]:
# Quick controls: run_all=True will run the entire job_queue (careful).
run_all = False
max_jobs = 2  # when run_all=False, only run first `max_jobs` entries

master_seed = grid['seeds'][0] if grid.get('seeds') else 0
generations = grid['generations']
repeats = grid.get('repeats', 30)

results = []
jobs_to_run = job_queue if run_all else job_queue[:max_jobs]

for i, (algo, task, dim, pop, unroll) in enumerate(jobs_to_run, 1):
    print(f'Running job {i}/{len(jobs_to_run)}: Algo={algo}, Task={task}, Dim={dim}, Pop={pop}, Unroll={unroll}')
    try:
        packaged = run_job(algo, task, dim, pop, unroll, master_seed, generations, repeats, hyperparams)
        results.extend(packaged)
    except Exception as e:
        print('ERROR running job:', e)

# Convert to DataFrame
df = pd.DataFrame(results)
df.head()

Running job 1/2: Algo=malthusjax, Task=rastrigin, Dim=20, Pop=100, Unroll=1
ERROR running job: Unknown algo: malthusjax. Available: ['Standard_GA']
Running job 2/2: Algo=malthusjax, Task=rastrigin, Dim=20, Pop=100, Unroll=25
ERROR running job: Unknown algo: malthusjax. Available: ['Standard_GA']


""


## 6) Save & inspect results
Save a CSV copy and display summary statistics.

In [ ]:
# Save results with timestamp
if not df.empty:
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_file = os.path.join(OUTPUT_DIR, f'notebook_benchmark_{ts}.csv')
    df.to_csv(out_file, index=False)
    print('Saved:', out_file)
    display(df.describe(include='all'))
else:
    print('No results to save (df is empty).')

No results to save (df is empty).


## Notes and next steps
- Use `run_all = True` to run the full grid (may be long).
- Adjust `grid` or `hyperparams` cells to explore different setups.
- The notebook mirrors `benchmarks/cli.py` but keeps execution interactive and educational.